# Urban Semantic & Instance Segmentation — Kaggle environment check

Este Notebook prepara el entorno remoto, clona la rama privada del proyecto y comprueba GPU, dependencias e inputs montados. No descarga datasets ni inicia entrenamiento.

In [ ]:
from pathlib import Path
import os, sys, subprocess, base64, json

WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'Procesamiento-de-Im-genes-'
RESULTS_DIR = WORK / 'urban-results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Working directory:', WORK)
print('Results directory:', RESULTS_DIR)

## 1. Clonar repositorio privado

Crea y habilita el secreto Kaggle `GITHUB_TOKEN` antes de ejecutar esta celda. Debe ser un Fine-grained token con Contents: Read-only para este repositorio.

In [ ]:
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Falta el secreto GITHUB_TOKEN. Añádelo en Add-ons → Secrets y habilítalo para este Notebook.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
if REPO_DIR.exists():
    print('Repositorio ya existe:', REPO_DIR)
else:
    cmd = ['git', '-c', f'http.extraheader=AUTHORIZATION: basic {auth}', 'clone',
           '--depth', '1', '--branch', 'urban-models-and-benchmark',
           'https://github.com/Leohf04/Procesamiento-de-Im-genes-.git', str(REPO_DIR)]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    print('Clonación completada.')
del token, auth
print('Repo:', REPO_DIR)
print('README existe:', (REPO_DIR / 'README.md').is_file())

## 2. Comprobar Python, PyTorch y GPU

La selección de GPU se realiza desde Settings → Session options. Esta celda solo verifica el dispositivo asignado.

In [ ]:
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPUs visibles:', torch.cuda.device_count())
else:
    print('AVISO: no hay GPU CUDA. Revisa Settings → Accelerator.')

## 3. Inspeccionar datasets adjuntos

Añade datasets desde Input → Add Input después de verificar procedencia, licencia, estructura y anotaciones. Esta celda solo lista archivos; no copia datos.

In [ ]:
INPUT_DIR = Path('/kaggle/input')
if not INPUT_DIR.exists():
    print('No existe /kaggle/input todavía.')
else:
    for item in sorted(INPUT_DIR.iterdir()):
        print(('DIR ' if item.is_dir() else 'FILE') + str(item))
        if item.is_dir():
            for child in sorted(item.iterdir())[:20]:
                print('   ', child.name + ('/' if child.is_dir() else ''))
print('\nNo se ha descargado ni procesado ningún dataset.')

## 4. Preparar directorio de resultados

Los checkpoints, métricas y manifiestos deben escribirse en `/kaggle/working/urban-results`. Guarda versiones/salidas periódicamente para no depender de una sesión interactiva.

In [ ]:
for name in ['checkpoints', 'metrics', 'logs', 'predictions']:
    (RESULTS_DIR / name).mkdir(exist_ok=True)
print('Directorios preparados:')
for p in sorted(RESULTS_DIR.iterdir()):
    print('-', p.name)
print('\nDiagnóstico terminado. No se ha iniciado entrenamiento.')